# DDM fit to value-capture pilot data (HSSM)

Fits a hierarchical Drift Diffusion Model to the singleton task.
Conditions: `absent`, `rank-0`, `rank-1`, `rank-2` (distractor value from low to high).

We compare:
- **Model 1 (v only)**: drift rate `v` varies by condition; `a`, `t` shared.
- **Model 2 (v + a)**: both `v` and `a` vary by condition; `t` shared.

Both models include per-subject random intercepts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hssm
import arviz as az
from pathlib import Path

hssm.set_floatX('float32')  # recommended for JAX/numpyro backend

BIDS_DIR = Path('/data/ds-valuecapture')
subjects = [1, 2]
sessions = [1, 2]
CONDITION_ORDER = ['absent', 'rank-0', 'rank-1', 'rank-2']

## Load and prepare data

In [ ]:
dfs = []
for sub in subjects:
    for ses in sessions:
        func_dir = BIDS_DIR / f'sub-{sub:02d}' / f'ses-{ses}' / 'func'
        for tsv in sorted(func_dir.glob(
                f'sub-{sub:02d}_ses-{ses}_task-valuecapture_run-*_events.tsv')):
            run = int(tsv.stem.split('run-')[1].split('_')[0])
            df = pd.read_csv(tsv, sep='\t')
            df['subject'] = sub
            df['session'] = ses
            df['run'] = run
            dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(dfs)} runs, {raw.subject.nunique()} subjects')

In [ ]:
# One row per trial (iti1 phase stores rt and correct)
trials = raw[raw['event_type'] == 'iti1'].copy()

def make_condition(row):
    if not row['distractor_present']:
        return 'absent'
    return f'rank-{int(row["value_rank"])}'

trials['condition'] = trials.apply(make_condition, axis=1)
trials['condition'] = pd.Categorical(
    trials['condition'], categories=CONDITION_ORDER, ordered=True)

# HSSM DDM expects:
#   rt       – response time in seconds (must be > 0)
#   response – 1 = upper boundary (correct), -1 = lower boundary (error)
data = trials[['subject', 'session', 'run', 'condition', 'rt', 'correct']].copy()
data = data.rename(columns={'correct': 'response'})

# Drop timeouts (no RT or no response recorded) before casting
data = data.dropna(subset=['rt', 'response'])
data = data[(data['rt'] > 0.1) & (data['rt'] < 3.0)]

# Recode: True/1 → 1 (upper), False/0 → -1 (lower)
data['response'] = data['response'].astype(int).map({1: 1, 0: -1})

# HSSM requires 'subj_idx' for hierarchical models
data['subj_idx'] = data['subject'].astype(str)

print(data.shape)
print(data['condition'].value_counts().sort_index())
print(data['response'].value_counts())
data.head()

## Quick EDA: RT distributions per condition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cond, grp in data.groupby('condition', observed=True):
    axes[0].hist(grp['rt'], bins=40, alpha=0.5, label=cond, density=True)
axes[0].set_xlabel('RT (s)')
axes[0].set_ylabel('Density')
axes[0].set_title('RT distributions by condition')
axes[0].legend()

# response is -1/1; accuracy = proportion correct (==1)
acc = data.groupby('condition', observed=True)['response'].apply(lambda x: (x == 1).mean())
acc.plot.bar(ax=axes[1], color='steelblue')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy by condition')
axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, ls='--', color='gray')

plt.tight_layout()
plt.show()

## Model 1: drift rate `v` varies by condition

Condition and subject as fixed effects on `v`; `a` and `t` shared.

(Random effects require many groups to estimate variance — with only 2 subjects, fixed effects are appropriate.)

In [ ]:
model_v = hssm.HSSM(
    data=data,
    model='ddm',
    include=[
        {
            'name': 'v',
            'formula': 'v ~ 1 + C(condition, Treatment("absent")) + subj_idx',
        },
    ],
)
model_v

In [ ]:
idata_v = model_v.sample(
    sampler='nuts_numpyro',
    chains=4,
    draws=1000,
    tune=2000,
    target_accept=0.95,
)

In [ ]:
az.plot_trace(idata_v, var_names=['~subj_idx'], compact=True)
plt.tight_layout()
plt.show()

In [ ]:
az.summary(idata_v, var_names=['~subj_idx'], round_to=3)

## Model 2: both `v` and `a` vary by condition

In [ ]:
model_va = hssm.HSSM(
    data=data,
    model='ddm',
    include=[
        {
            'name': 'v',
            'formula': 'v ~ 1 + C(condition, Treatment("absent")) + subj_idx',
        },
        {
            'name': 'a',
            'formula': 'a ~ 1 + C(condition, Treatment("absent")) + subj_idx',
        },
    ],
)
model_va

In [ ]:
idata_va = model_va.sample(
    sampler='nuts_numpyro',
    chains=4,
    draws=1000,
    tune=2000,
    target_accept=0.95,
)

In [ ]:
az.plot_trace(idata_va, var_names=['~subj_idx'], compact=True)
plt.tight_layout()
plt.show()

In [ ]:
az.summary(idata_va, var_names=['~subj_idx'], round_to=3)

## Model comparison (WAIC / LOO)

In [ ]:
comparison = az.compare(
    {'v_only': idata_v, 'v_and_a': idata_va},
    ic='loo',
    scale='log',
)
comparison

In [ ]:
az.plot_compare(comparison, insample_dev=False)
plt.tight_layout()
plt.show()

## Posterior predictive check (winning model)

In [ ]:
# Change to model_v / idata_v if that wins
winning_model = model_va
winning_idata = idata_va

# Subsample posterior to keep PPC fast (200 draws × 4 chains → 50 per chain)
idata_thin = winning_idata.sel(draw=slice(None, None, winning_idata.posterior.dims['draw'] // 50))

ppc = winning_model.sample_posterior_predictive(idata=idata_thin)

# Plot observed vs predicted RT quantiles per condition
quantiles = [0.1, 0.3, 0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, len(CONDITION_ORDER), figsize=(14, 4), sharey=True)

# PPC RTs are signed: positive = correct (upper), negative = error (lower)
ppc_rt = ppc.posterior_predictive['rt'].values  # (chain, draw, obs)
ppc_rt_flat = ppc_rt.reshape(-1, ppc_rt.shape[-1])  # (samples, obs)

for ax, cond in zip(axes, CONDITION_ORDER):
    mask = (data['condition'] == cond).values

    obs_q = np.quantile(data.loc[mask & (data['response'] == 1), 'rt'], quantiles)
    pred_correct = ppc_rt_flat[:, mask]
    pred_correct = pred_correct[pred_correct > 0].reshape(-1)  # keep upper-boundary draws

    ax.fill_between(
        quantiles,
        np.quantile(pred_correct.reshape(ppc_rt_flat.shape[0], -1), 0.025, axis=0)
        if pred_correct.size > 0 else np.full(len(quantiles), np.nan),
        np.quantile(pred_correct.reshape(ppc_rt_flat.shape[0], -1), 0.975, axis=0)
        if pred_correct.size > 0 else np.full(len(quantiles), np.nan),
        alpha=0.3, label='95% CI',
    )
    ax.scatter(quantiles, obs_q, color='black', zorder=5, label='observed')
    ax.set_title(cond)
    ax.set_xlabel('Quantile')

axes[0].set_ylabel('RT (s)')
axes[0].legend(fontsize=8)
plt.suptitle('Posterior predictive: RT quantiles (correct trials)', y=1.02)
plt.tight_layout()
plt.show()

## Forest plot: condition effects on `v` (and `a`)

In [ ]:
# Population-level (fixed) effects
pop_vars_v = [v for v in winning_idata.posterior.data_vars
              if v.startswith('v_') and 'subj_idx' not in v]
pop_vars_a = [v for v in winning_idata.posterior.data_vars
              if v.startswith('a_') and 'subj_idx' not in v]

print('v params:', pop_vars_v)
print('a params:', pop_vars_a)

az.plot_forest(
    winning_idata,
    var_names=pop_vars_v + pop_vars_a,
    combined=True,
    hdi_prob=0.94,
)
plt.axvline(0, ls='--', color='gray')
plt.tight_layout()
plt.show()